[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/quantitative_imaging_molli.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Quanitative Image Reconstruction - T1 Mapping using MOLLI Sequence

In [ ]:
import numpy as np
import torch
from pathlib import Path
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import MOLLI
from cmap import Colormap

### Download data

The following zenodo data sets can be used with this notebook:
- [IBT-CMR (Zurich, Switzerland), Philips Ingenia Ambition X, ramped down to 0.6T](https://zenodo.org/records/18847561)

In [ ]:
# Get the raw data from zenodo
import os
import tempfile
import zipfile
import zenodo_get

tmp = tempfile.TemporaryDirectory()  # RAII, automatically cleaned up
data_folder = Path(tmp.name)
zenodo_get.download(
    record="18460613",
    retry_attempts=5,
    output_dir=data_folder,
    file_glob=("Subject 1.zip",),
)
with zipfile.ZipFile(data_folder / Path("Subject 1.zip"), "r") as zip_ref:
    zip_ref.extractall(data_folder)

# List all files
for root, dirs, files in os.walk(data_folder):
    for name in files:
        print(os.path.join(root, name))

## Select the raw data files

In [ ]:
raw_files = list(
    (
        data_folder
        / Path("Subject 4/Quantitative protocol/Subject4-RightKnee-T1map/ismrmrd/")
    ).rglob("*.h5")
)
image_folder = data_folder / Path(
    "Subject 4/Quantitative protocol/Subject4-RightKnee-T1map/Dicom/"
)

rsa_or_spr = "rsa"  # 'rsa' for knee imaging, 'spr' for brain imaging
vmax = 1.0
ti_times = 0.340 + torch.as_tensor([0, 1, 2, 3, 0, 1, 2, 0, 1, 0, 1])

## Load Dicom

In [ ]:
t1_map_dicom = IData.from_dicom_folder(image_folder)

# Dicom saves T1 values in ms, we use seconds here.
t1_map_dicom.data *= 1e-3

## Reconstruct images

In [ ]:
kdata = [
    KData.from_file(fname, trajectory=KTrajectoryCartesian()) for fname in raw_files
]
kdata = kdata[0].concatenate(*kdata[1:], dim=0)

# for multi-slice acquisition: resort the data into correct order of slices and contrasts
if (nslices := kdata.header.acq_info.idx.slice.unique().numel()) > 1:
    positions = torch.stack(
        torch.broadcast_tensors(*kdata.header.acq_info.position.zyx), -1
    ).squeeze()
    orientation = torch.stack(
        torch.broadcast_tensors(
            *kdata.header.acq_info.orientation.as_directions()[0].zyx
        )
    )
    orientation = orientation.squeeze()
    if orientation.ndim == 2:
        orientation = orientation[..., 0]
    sort_idx = torch.argsort((positions @ orientation).squeeze(), stable=True)
    kdata = kdata[sort_idx].rearrange(
        "(slice contrast) ... -> contrast slice ...", slice=nslices
    )


# We have to calculate the coil maps from one of the contrasts which ideally has high signal for all tissue types
csm = CsmData.from_kdata_inati(kdata[0])
recon = DirectReconstruction(kdata, csm=csm)
idata = recon(kdata)

if ti_times is None:
    ti_times = idata.header.ti

## Create signal model for MOLLI sequence and calculate quantitative maps

We will select the correct signal model and then create a dictionary mapping operator, similarly what is used in MR Fingerprinting. 
We can then simply apply the dictionary mapping operator to the reconstructed images and obtain the quantitative maps

In [ ]:
dictionary = DictionaryMatchOp(
    MOLLI(ti=torch.as_tensor(ti_times, dtype=torch.float32)),
    index_of_scaling_parameter=0,
)
dictionary.append(
    torch.tensor(1.0),
    torch.linspace(0.1, 3.0, 1000)[None, :],
    torch.linspace(0.1, 5.0, 1000)[None, None, :],
)
m0_match, c_match, t1_match = dictionary(idata.data)
t1_map = IData(t1_match, header=idata.header[0])

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from einops import rearrange
import nibabel as nib
from typing import Literal

from nibabel.orientations import (
    io_orientation,
    axcodes2ornt,
    ornt_transform,
    apply_orientation,
)


def show_image(
    qmap: IData, cmap, vmax: float, rsa_or_spr: Literal["rsa", "spr"] = "rsa"
) -> None:
    """Show the qualitative images."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    for cax in ax.flatten():
        cax.set_axis_off()

    def orient_images(idata: IData) -> np.array:
        orientation = idata.header.orientation.as_matrix().squeeze()
        if orientation.ndim == 3:
            orientation = orientation.mean(0)
        affine_zyx = torch.cat(
            [
                torch.tensor([[1.0, 0.0, 0.0, 0.0]]),
                torch.cat([torch.zeros((3, 1)), orientation], 1),
            ],
            0,
        )

        data = rearrange(idata.data, "... other z y x-> x y z 1 other (...)")
        img_nii = nib.Nifti1Image(
            data.squeeze().abs().numpy(force=True),
            affine_zyx.flip([0, 1]).numpy(),
            dtype=np.float32,
        )
        # Target orientation (RAS)
        ras_ornt = axcodes2ornt(tuple((rsa_or_spr.upper())))
        transform = ornt_transform(io_orientation(img_nii.affine), ras_ornt)
        ras_data = apply_orientation(img_nii.get_fdata(), transform)
        return ras_data

    def plot_multi_slice_image(ax, img, colorbar_label, cmap, vmax):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        idim = img.squeeze().shape[0]
        center = idim // 2
        offset = max(1, idim // 6)
        idx = np.clip([center - offset, center, center + offset], 0, idim - 1)
        for cax, slice_idx in zip(ax, idx):
            im = cax.imshow(
                np.squeeze(img[slice_idx, :, :]),
                cmap=cmap,
                vmin=0,
                vmax=vmax,
                origin="lower",
            )
            fig.colorbar(im, ax=cax, label=colorbar_label)

    plot_multi_slice_image(ax, orient_images(qmap), "T1 (s)", cmap, vmax)

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(t1_map, Colormap("lipari").to_mpl(), vmax, rsa_or_spr)
show_image(t1_map_dicom, Colormap("lipari").to_mpl(), vmax, rsa_or_spr)